In [0]:
# Biblioteca para extrair os arquivos .7z do Novo Caged
# Execute esta célula antes das demais em cada nova sessão
%pip install py7zr -q

In [0]:
from pyspark.sql.functions import *
import os
import re
import time
import py7zr
import pandas as pd

In [0]:
# Caminho do volume com os arquivos brutos (.7z) do Novo Caged,
# organizados em subdiretórios por ano (2021, 2022, ...)
path = "/Volumes/workspace/bronze/raw_novo_caged"

# Diretório de staging para a extração temporária dos .txt
dir_extracao = f"{path}/_extracao"

# Tabela Bronze registrada no Unity Catalog
tabela_bronze = "workspace.bronze.novo_caged_movimentacoes"

# Competência usada na demonstração passo a passo das próximas células
# (a carga completa de todas as competências está na célula "Processar Todas")
competencia = "202101"

# Filtro para a carga completa:
#   None                 -> processa todas as competências (carga manual completa)
#   []                   -> nada a processar (a verificação do 00 não achou novidades)
#   ["202103", "202104"] -> processa apenas as competências informadas
#
# Prioridade de resolução:
#   1) widget "competencias_processar" (use TODAS para forçar a carga completa)
#   2) arquivo de controle escrito pelo notebook 00_verificar_novos_arquivos
#      (resultado da verificação do FTP na tarefa anterior do job)
#   3) nenhuma fonte -> processa todas
dbutils.widgets.text("competencias_processar", "")
_parametro_competencias = dbutils.widgets.get("competencias_processar").strip()

arquivo_controle = f"{path}/_controle/novas_competencias.txt"

if _parametro_competencias.upper() == "TODAS":
    competencias_processar = None
elif _parametro_competencias == "" and os.path.exists(arquivo_controle):
    with open(arquivo_controle, encoding="utf-8") as arq_controle:
        _resultado_verificacao = arq_controle.read().strip()
    if _resultado_verificacao == "NENHUMA":
        # O 00 verificou o FTP e não há novidades: nada a processar
        competencias_processar = []
    else:
        competencias_processar = [
            comp.strip()
            for comp in _resultado_verificacao.split(",")
            if comp.strip()
        ]
elif _parametro_competencias == "":
    competencias_processar = None
else:
    competencias_processar = [
        comp.strip()
        for comp in _parametro_competencias.split(",")
        if comp.strip()
    ]

In [0]:
# Se a verificação do notebook 00 não encontrou novas competências,
# encerra a ingestão: em um job, a tarefa conclui com sucesso e o job termina
# sem executar as demais células. Para uma carga manual completa, use o
# widget "competencias_processar" com o valor TODAS.
if competencias_processar == []:
    print("Nenhuma nova competência para ingerir. Encerrando a ingestão.")
    dbutils.notebook.exit("Nenhuma nova competência para ingerir.")

print(
    "Competências a processar nesta execução:",
    "todas"
    if competencias_processar is None
    else ", ".join(competencias_processar),
)

In [0]:
# Localiza todos os arquivos .7z no volume e deriva a competência a partir
# do nome do arquivo. Ex.: .../2021/CAGEDMOV202101.7z -> competência 202101
arquivos_caged = []

for raiz, _, nomes in os.walk(path):
    # Ignora o diretório de staging da extração
    if raiz.startswith(dir_extracao):
        continue
    for nome in nomes:
        resultado = re.match(r"CAGEDMOV(\d{6})\.7z$", nome, re.IGNORECASE)
        if resultado:
            arquivos_caged.append((resultado.group(1), os.path.join(raiz, nome)))

arquivos_caged.sort()

assert len(arquivos_caged) > 0, "Nenhum arquivo .7z encontrado no volume."

anos = sorted({comp[:4] for comp, _ in arquivos_caged})

print(f"Arquivos encontrados: {len(arquivos_caged)}")
print(f"Anos disponíveis: {', '.join(anos)}")
print(f"Primeira competência: {arquivos_caged[0][0]}")
print(f"Última competência:  {arquivos_caged[-1][0]}")

In [0]:
# Seleciona o .7z da competência de demonstração
arquivo_7z = next(caminho for comp, caminho in arquivos_caged if comp == competencia)

# Extrai o .7z para o diretório de staging
# (o formato 7z não é lido nativamente pelo Spark)
destino_extracao = f"{dir_extracao}/{competencia}"
os.makedirs(destino_extracao, exist_ok=True)

with py7zr.SevenZipFile(arquivo_7z, mode="r") as z:
    z.extractall(path=destino_extracao)

# Localiza o .txt extraído
arquivo_txt = next(
    os.path.join(raiz, nome)
    for raiz, _, nomes in os.walk(destino_extracao)
    for nome in nomes
    if nome.lower().endswith(".txt")
)

print(f"Arquivo extraído: {arquivo_txt}")

# Lê o arquivo .txt com delimitador ";" e usando a primeira linha como cabeçalho
df = (
    spark
    .read
    .format("csv")
    .option("delimiter", ";")
    .option("encoding", "UTF-8")
    .option("header", True)
    .option("inferSchema", False)
    .load(arquivo_txt))

display(df)

In [0]:
df_bronze = (
    df
    .withColumn("_competencia_referencia", lit(competencia))
    .withColumn("_arquivo_origem", lit(os.path.basename(arquivo_7z)))
    .withColumn("_caminho_origem", lit(arquivo_7z))
    .withColumn("_data_ingestao", current_timestamp())
)

print(f"Total de colunas: {len(df_bronze.columns)}")

df_bronze.select(
    "competênciamov",
    "uf",
    "cbo2002ocupação",
    "saldomovimentação",
    "_competencia_referencia",
    "_arquivo_origem",
    "_data_ingestao"
).show(5, truncate=False)

In [0]:
# Quantidade total de registros
total_registros = df_bronze.count()

# Quantidade de registros com competência divergente ou nula
registros_invalidos = df_bronze.filter(
    col("competênciamov").isNull()
    | (col("competênciamov") != competencia)
    | col("_competencia_referencia").isNull()
    | (col("_competencia_referencia") != competencia)
).count()

# Validações obrigatórias
assert total_registros > 0, (
    "O arquivo não contém registros."
)

assert registros_invalidos == 0, (
    f"Foram encontrados {registros_invalidos} registros "
    f"com competência inválida para {competencia}."
)

print("Validação concluída com sucesso!")
print(f"Competência: {competencia}")
print(f"Total de registros: {total_registros:,}")
print(f"Registros inválidos: {registros_invalidos:,}")

In [0]:
# Proteção contra gravação de competência incorreta
assert total_registros > 0, "Não existem registros para gravar."
assert registros_invalidos == 0, "Existem registros inválidos."

# Sobrescrever somente a competência selecionada
# Na primeira execução a tabela ainda não existe: criar antes de sobrescrever
# O liquid clustering por competência torna as sobrescritas eficientes
tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.bronze LIKE 'novo_caged_movimentacoes'")
    .count() > 0
)

if tabela_existe:
    (
        df_bronze.writeTo(tabela_bronze)
        .using("delta")
        .overwrite(
            col("_competencia_referencia") == competencia
        )
    )
else:
    (
        df_bronze.writeTo(tabela_bronze)
        .using("delta")
        .clusterBy("_competencia_referencia")
        .create()
    )

print(f"Competência {competencia} reprocessada com sucesso!")

In [0]:
# Carga completa: extrai, valida e grava todas as competências do volume.
# Cada competência é sobrescrita isoladamente (replaceWhere), o que torna
# a execução idempotente: reprocessar um mês não afeta os demais.
# Para retomar uma carga interrompida, use o filtro "competencias_processar".
arquivos_processar = [
    (comp, caminho)
    for comp, caminho in arquivos_caged
    if competencias_processar is None or comp in competencias_processar
]

tabela_existe = (
    spark.sql("SHOW TABLES IN workspace.bronze LIKE 'novo_caged_movimentacoes'")
    .count() > 0
)

print(f"Competências a processar: {len(arquivos_processar)}")

inicio = time.time()
for indice, (comp, caminho_7z) in enumerate(arquivos_processar, start=1):
    t0 = time.time()

    # 1) Extrai o .7z para o diretório de staging
    destino = f"{dir_extracao}/{comp}"
    os.makedirs(destino, exist_ok=True)
    with py7zr.SevenZipFile(caminho_7z, mode="r") as z:
        z.extractall(path=destino)

    arquivo_txt = next(
        os.path.join(raiz, nome)
        for raiz, _, nomes in os.walk(destino)
        for nome in nomes
        if nome.lower().endswith(".txt")
    )

    # 2) Lê o .txt e adiciona os metadados de ingestão
    df_comp = (
        spark
        .read
        .format("csv")
        .option("delimiter", ";")
        .option("encoding", "UTF-8")
        .option("header", True)
        .option("inferSchema", False)
        .load(arquivo_txt)
        .withColumn("_competencia_referencia", lit(comp))
        .withColumn("_arquivo_origem", lit(os.path.basename(caminho_7z)))
        .withColumn("_caminho_origem", lit(caminho_7z))
        .withColumn("_data_ingestao", current_timestamp())
    )

    # 3) Validação em uma única leitura (total + registros inválidos)
    resumo_val = (
        df_comp
        .agg(
            count(lit(1)).alias("total"),
            count(
                when(
                    col("competênciamov").isNull()
                    | (col("competênciamov") != comp)
                    | col("_competencia_referencia").isNull()
                    | (col("_competencia_referencia") != comp),
                    lit(1),
                )
            ).alias("invalidos"),
        )
        .collect()[0]
    )

    assert resumo_val["total"] > 0, f"{comp}: arquivo sem registros."
    assert resumo_val["invalidos"] == 0, (
        f"{comp}: {resumo_val['invalidos']} registros com competência inválida."
    )

    # 4) Grava na Bronze (cria a tabela na primeira execução)
    if tabela_existe:
        (
            df_comp.writeTo(tabela_bronze)
            .using("delta")
            .overwrite(col("_competencia_referencia") == comp)
        )
    else:
        (
            df_comp.writeTo(tabela_bronze)
            .using("delta")
            .clusterBy("_competencia_referencia")
            .create()
        )
        tabela_existe = True

    # 5) Remove o .txt extraído para não acumular dados no volume
    os.remove(arquivo_txt)
    os.rmdir(destino)

    print(
        f"[{indice}/{len(arquivos_processar)}] {comp}: "
        f"{resumo_val['total']:,} registros em {time.time() - t0:.0f}s"
    )

print(f"\nCarga concluída em {(time.time() - inicio) / 60:.1f} minutos")

In [0]:
# Validar a tabela Bronze consolidada após a carga completa
df_validacao = spark.table(tabela_bronze)

# Registros por competência
display(
    df_validacao
    .groupBy("_competencia_referencia")
    .agg(count(lit(1)).alias("registros"))
    .orderBy("_competencia_referencia")
)

total_bronze = df_validacao.count()

# Nenhuma competência pode ter ficado vazia na tabela final
competencias_vazias = (
    df_validacao
    .groupBy("_competencia_referencia")
    .agg(count(lit(1)).alias("registros"))
    .filter(col("registros") == 0)
    .count()
)
assert competencias_vazias == 0, "Existem competências sem registros na Bronze."

print("Validação da Bronze concluída!")
print(f"Total de registros na Bronze: {total_bronze:,}")
print(f"Quantidade de colunas: {len(df_validacao.columns)}")

# Consultar o histórico de transações Delta
display(
    spark.sql(f"DESCRIBE HISTORY {tabela_bronze}")
    .select("version", "timestamp", "operation", "operationParameters")
    .limit(10)
)